# Faithfulness e-SNLI — Gemma3-27b-it with SAE Activation Analysis

In [ ]:
import sys, os

# Add LASR-main project root so `src.*` imports work.
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
from src.configs import ModelConfig, InferenceConfig, PromptStyle, SAEConfig, DatasetConfig
from src.dataset.esnli import ESNLI_Dataset
from src.gemma_model import GemmaModel
from src.SAE import JumpReLUSAE

# Configuration

In [ ]:
model_config = ModelConfig(model_name="google/gemma-3-27b-it")
inference_config = InferenceConfig(batch_size=2, max_new_tokens=256, downsample_rate=100)
prompt_style = PromptStyle.CHAIN_OF_THOUGHT_TAGS
use_few_shot = False

sae_config = SAEConfig(
    repo_id="google/gemma-scope-2-27b-it",
    sae_type="resid_post",
    layer=40,
    width="65k",
    l0="medium",
)

dataset_config = DatasetConfig(
    path="esnli/esnli",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "validation"},
    few_shot=False,
)

print(f"Model:      {model_config.model_name}")
print(f"Device:     {model_config.device}")
print(f"Batch size: {inference_config.batch_size}")
print(f"Downsample: 1/{inference_config.downsample_rate}")
print(f"Prompt:     {prompt_style.value}")
print(f"Few-shot:   {use_few_shot}")
print(f"SAE layer:  {sae_config.layer}")
print(f"SAE width:  {sae_config.width}")
print(f"SAE l0:     {sae_config.l0}")
print(f"SAE repo:   {sae_config.repo_id}")

# Setup — HF Token

In [ ]:
import sys
from huggingface_hub import login

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

# Data — Load e-SNLI

In [ ]:
esnli_dataset = ESNLI_Dataset(dataset_config)
esnli_df = esnli_dataset.get_sample_dataframe(n=len(esnli_dataset))
esnli_df.head()

# Build Prompts

In [ ]:
prompted_data = esnli_dataset.build_prompts()
esnli_df = prompted_data.to_pandas()
print(esnli_df["prompt"].iloc[0])

# Load Model + SAE

In [ ]:
model = GemmaModel(model_config)
tokenizer = model.tokenizer
sae = JumpReLUSAE.from_pretrained(sae_config, device=model_config.device)

## Generate and Gather Activations

In [ ]:
import textwrap

# Pick one sample prompt and generate
sample = esnli_df.sample(1)
sample_prompt = sample["prompt"].item()
sample_label = sample["gold_label"].item()

generation, full_ids, prompt_len = model.generate(
    sample_prompt, max_new_tokens=inference_config.max_new_tokens)

gen_len = full_ids.shape[1] - prompt_len
print(f"{'PROMPT LENGTH':=^80}")
print(f"Prompt tokens length: {prompt_len}")
print(f"Generated tokens length: {gen_len}")
print(f"Full sequence tokens length: {full_ids.shape[1]}")

print(f"{'FULL SEQUENCE':=^80}")
wrapper = textwrap.TextWrapper(width=80)
print("\n".join(wrapper.fill(line) for line in generation.splitlines()))
print(f"Actual label: {sample_label}")

# Gather residual activations at SAE target layer (full sequence)
print(f"{'RESIDUAL ACTIVATIONS':=^80}")
residual_acts = model.gather_residual_activations(sae_config.layer, full_ids[0])
# residual_acts shape: (n_tokens, d_model)
print(f"Residual activations shape: {residual_acts.shape}")

# Reconstruction & sparsity metrics on full prompt (reported once)
stats = sae.get_reconstruction_stats(residual_acts.float())
print(f"FVU: {stats['fvu']:.2%}")
print(f"L0:  {stats['l0']:.1f}")

# Encode generated-only slice
print(f"{'SAE FEATURE ACTIVATIONS':=^80}")
gen_acts = residual_acts[prompt_len:]
sae_acts_gen = sae.encode(gen_acts.float())
# sae_acts_gen shape: (gen_tokens, n_features)
print(f"SAE activations (gen-only): {sae_acts_gen.shape}")

# Also encode full sequence for later use
sae_acts_full = sae.encode(residual_acts.float())
print(f"SAE activations (full):     {sae_acts_full.shape}")

# Build token strings
all_tokens = tokenizer.convert_ids_to_tokens(full_ids[0])
gen_tokens = all_tokens[prompt_len:]
print(f"All tokens: {len(all_tokens)}, Generated tokens: {len(gen_tokens)}")

# Feature Summary

In [10]:
# Top-K features per token (generation-only slice)
from neuronpedia_client import build_sae_id
from lasr.feature import create_features

def summarize_sae_features(
    sae_activations: torch.Tensor,
    tokens: list[str],
    top_k: int = 10,
    print_first_n: int = 3) -> None:
    """Summarize top-k SAE features per token.

    Prints the top-k features for first 3 tokens and plots the heatmap for all
    tokens in list.

    Args:
        sae_activations:
            SAE activation tensor of shape ``(1, n_tokens, n_features)``.
        top_k:
            Number of top features to summarize per token.
        tokens:
            List of token strings, same length as ``sae_activations`` dim 1.
    """
    per_token_vals, per_token_idxs = top_k_features_per_token(
        sae_activations, k=top_k)

    # First 6 tokens are formatting tokens.
    for n in range(print_first_n):
        print(f"\nTop {top_k} SAE features per token (first 3 tokens shown):")
        for t in range(6, min(6+n, len(tokens))):
            print(f"\nToken '{tokens[t]}':")
            for r in range(top_k):
                idx = int(per_token_idxs[t, r])
                val = float(per_token_vals[t, r])
                print(
                    f"  Rank {r+1}  feature {idx:>5d}  activation = {val:.4f}")

    np_model_id = "gemma-3-27b-it"
    np_sae_id = build_sae_id(sae_config)
    unique_features = sorted(
        set(per_token_idxs.cpu().numpy().ravel().tolist()))
    features = create_features(sae_acts_full, unique_features, all_tokens)

    # Fetch Neuronpedia details
    for f in features:
        f.fetch_details(np_model_id, np_sae_id)

    feature_map = {f.feature_idx: f for f in features}
    print(f"Created {len(features)} Feature objects")
    print(f"Example: {feature_map[unique_features[0]]!r}")

    # Build labels dict for the heatmap
    labels = {f.feature_idx: f.label for f in features}

    fig = plot_per_token_topk_heatmap(
        per_token_vals,
        per_token_idxs,
        tokens=tokens,
        labels=labels,
        title="Gemma3-27b-it Per-Token Top-K SAE Feature Activations (Layer 40)",
    )
    fig.show()

    return feature_map


## Raw Features

In [11]:
# Generated tokens raw activations.
gen_token_ids = full_ids[0, prompt_len:]
tokens = tokenizer.convert_ids_to_tokens(gen_token_ids)

raw_feature_map = summarize_sae_features(sae_acts_gen, tokens, top_k=10, print_first_n=0)

Created 556 Feature objects
Example: Feature(idx=8, label='evaluating complexity and worth', max_act=3222.61, n_tokens=249)


In [12]:
# --- Inspect features using the Feature class ---
example_feature = int(1383)
print(f"Inspecting feature {example_feature} (top feature for first generated token)")
print(f"repr: {raw_feature_map[example_feature]!r}")
print(f"frac_nonzero: {raw_feature_map[example_feature].frac_nonzero}")
print(f"top_tokens: {raw_feature_map[example_feature].top_tokens()}")
print()

# Generation-only view
raw_feature_map[example_feature].inspect(token_range="generation", prompt_length=prompt_len)

Inspecting feature 1383 (top feature for first generated token)
repr: Feature(idx=1383, label=None, max_act=6213.60, n_tokens=249)
frac_nonzero: 0.04011021922994267
top_tokens: [('.', 6213.6005859375), ('.', 5766.3740234375), ('.', 5405.119140625), ('.', 4703.0244140625), ('.', 3926.359375)]



## Denoised Features

In [ ]:
from lasr.feature import create_features
from lasr.denoising import DenoisingConfig, denoise

# Denoise SAE activations (TF-IDF weighting)
# Denoising happens on the full sequence activations.
denoising_config = DenoisingConfig()  # defaults to continuous_tfidf
denoised_sae_acts_full, scaling_factors = denoise(
    sae_acts_full, denoising_config)

# Extract the generation-only slice.
denoised_sae_acts_gen = denoised_sae_acts_full[:, prompt_len:, :]
denoised_feature_map = summarize_sae_features(
    denoised_sae_acts_gen, tokens, top_k=50, print_first_n=0)

Created 776 Feature objects
Example: Feature(idx=8, label='evaluating complexity and worth', max_act=3222.61, n_tokens=249)


In [14]:
dn_per_token_vals, dn_per_token_idxs = top_k_features_per_token(
    denoised_sae_acts_gen, k=10)
print(dn_per_token_vals.min())

tensor(2264.7427, device='cuda:0')


In [15]:
example_feature = int(9)
print(f"Inspecting feature {example_feature} (top feature for first generated token)")
print(f"repr: {denoised_feature_map[example_feature]!r}")
print(f"frac_nonzero: {denoised_feature_map[example_feature].frac_nonzero}")
print(f"top_tokens: {denoised_feature_map[example_feature].top_tokens()}")
print()

# Generation-only view
denoised_feature_map[example_feature].inspect(token_range="generation", prompt_length=prompt_len)

Inspecting feature 9 (top feature for first generated token)


KeyError: 9

In [ ]:
scaling_factors[1228]

tensor(-2.5815, device='cuda:0')

In [ ]:
sae_acts_full.squeeze(0)[:,1228]

tensor([   0.0000,    0.0000,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000,    0.0000,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000,    0.0000,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000,    0.0000,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000,    0.0000,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000,    0.0000,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000,    0.0000,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000,    0.0000,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000,    0.0000,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000,    0.0000,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000,    0.0000,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000,    0.0000,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000,    0.0000,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000,    0.0000,    0.000

In [ ]:
denoised_sae_acts_full.squeeze(0)[:,1228]

tensor([   -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,
           -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,
           -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,
           -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,
           -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,
           -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,
           -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,
           -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,
           -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,
           -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,
           -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,
           -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,    -0.0000,
           -0.0000,    -0.0000,    -0.00